<a href="https://colab.research.google.com/github/maniratnam256/ai-mentor-portfolio/blob/main/Day2_ResumeExtractor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q google-genai pydantic
import os, getpass
if 'GEMINI_API_KEY' not in os.environ:
    os.environ['GEMINI_API_KEY'] = getpass.getpass('Gemini API key: ')

Gemini API key: ··········


In [2]:
from pydantic import BaseModel
from typing import List, Optional

class Education(BaseModel):
    degree: str
    institution: str
    year: int

class Resume(BaseModel):
    name: str
    email: str
    phone: Optional[str] = None
    education: List[Education]
    skills: List[str]
    projects: List[str] = []
    experience_years: float


In [3]:
from google import genai
from pydantic import ValidationError

client = genai.Client(api_key=os.environ['GEMINI_API_KEY'])

def extract_resume(raw_text: str, max_retries: int = 1) -> Resume:
    for attempt in range(max_retries + 1):
        try:
            resp = client.models.generate_content(
                model='gemini-2.5-flash',
                contents=f'Extract a Resume JSON from this text. Return ONLY JSON, no markdown.\n\n{raw_text}',
                config={
                    'response_mime_type': 'application/json',
                    'response_schema': Resume.model_json_schema(),
                },
            )
            return Resume.model_validate_json(resp.text)
        except ValidationError as e:
            if attempt == max_retries:
                raise
            # Retry once with the broken JSON in the prompt
            fix_prompt = (f'Fix this JSON to match schema. Errors: {e}. '
                          f'Original: {resp.text}')
            resp = client.models.generate_content(
                model='gemini-2.5-flash', contents=fix_prompt,
                config={
                    'response_mime_type': 'application/json',
                    'response_schema': Resume.model_json_schema(),
                },
            )
            return Resume.model_validate_json(resp.text)


In [14]:
# Load sample résumés from the lab kit
with open('./sample_data/sample_resumes.txt') as f:
    resumes = [r.strip() for r in f.read().split('---') if r.strip()]

print(f'Loaded {len(resumes)} sample résumés')

results = []
for i, r in enumerate(resumes[:3]):
    try:
        parsed = extract_resume(r)
        results.append(parsed)
        print(f'\nRésumé {i+1}: {parsed.name} — {len(parsed.skills)} skills, '
              f'{parsed.experience_years} years exp')
    except Exception as e:
        print(f'\nRésumé {i+1}: FAILED — {type(e).__name__}: {str(e)[:200]}')

# Print full first result
if results:
    print('\n=== Full first result ===')
    print(results[0].model_dump_json(indent=2))

Loaded 5 sample résumés

Résumé 1: Ravi Kumar — 6 skills, 0.25 years exp

Résumé 2: Sneha Reddy — 6 skills, 0.115 years exp

Résumé 3: Arun Pillai — 8 skills, 0.5 years exp

=== Full first result ===
{
  "name": "Ravi Kumar",
  "email": "ravi.kumar@example.com",
  "phone": "+91-9876543210",
  "education": [
    {
      "degree": "B.Tech Computer Science",
      "institution": "Aditya University",
      "year": 2026
    },
    {
      "degree": "Intermediate",
      "institution": "Narayana Junior College",
      "year": 2022
    }
  ],
  "skills": [
    "Python",
    "Java",
    "SQL",
    "Git",
    "Linux",
    "REST APIs"
  ],
  "projects": [
    "Built a Flask REST API for college placement portal",
    "Solved 250+ DSA problems on LeetCode",
    "ML model for crop yield prediction"
  ],
  "experience_years": 0.25
}


In [15]:
# Empty string — should fail gracefully, not crash
try:
    bad = extract_resume('')
    print('Unexpected success:', bad.model_dump_json())
except Exception as e:
    print('Caught gracefully:', type(e).__name__)
    print('Message:', str(e)[:200])


Unexpected success: {"name":"","email":"","phone":null,"education":[],"skills":[],"projects":[],"experience_years":0.0}


In [16]:
from pydantic import BaseModel
from typing import List, Optional
from google import genai
from pydantic import ValidationError
import os
import re


# -----------------------------
# Pydantic Models for MCQs
# -----------------------------

class MCQ(BaseModel):
    question: str
    options: List[str]
    correct_answer: Optional[str] = None


class MCQTest(BaseModel):
    title: str
    total_questions: int
    mcqs: List[MCQ]


# -----------------------------
# Gemini Client
# -----------------------------

client = genai.Client(api_key=os.environ['GEMINI_API_KEY'])


# -----------------------------
# Extract MCQs Function
# -----------------------------

def extract_mcqs(raw_text: str, max_retries: int = 1) -> MCQTest:

    for attempt in range(max_retries + 1):

        try:

            resp = client.models.generate_content(
                model='gemini-2.5-flash',

                contents=f"""
Extract MCQs from the following text.

Return ONLY valid JSON.

Rules:
- Extract title
- Extract all questions
- Extract options
- If answer is missing, set correct_answer as null

Text:
{raw_text}
""",

                config={
                    'response_mime_type': 'application/json',
                    'response_schema': MCQTest.model_json_schema(),
                },
            )

            return MCQTest.model_validate_json(resp.text)

        except ValidationError as e:

            if attempt == max_retries:
                raise

            fix_prompt = f"""
Fix this JSON to match schema.

Errors:
{e}

Broken JSON:
{resp.text}
"""

            resp = client.models.generate_content(
                model='gemini-2.5-flash',

                contents=fix_prompt,

                config={
                    'response_mime_type': 'application/json',
                    'response_schema': MCQTest.model_json_schema(),
                },
            )

            return MCQTest.model_validate_json(resp.text)


# -----------------------------
# Load MCQ File
# -----------------------------

with open('./sample_data/mcqs.txt', 'r', encoding='utf-8') as f:
    mcq_text = f.read()

print("MCQ file loaded successfully")


# -----------------------------
# Process MCQ File
# -----------------------------

try:

    parsed_mcqs = extract_mcqs(mcq_text)

    print("\n=== TEST DETAILS ===")
    print("Title:", parsed_mcqs.title)
    print("Total Questions:", parsed_mcqs.total_questions)

    print("\n=== SAMPLE QUESTIONS ===")

    for i, q in enumerate(parsed_mcqs.mcqs[:3], start=1):

        print(f"\nQuestion {i}:")
        print(q.question)

        for opt in q.options:
            print(opt)

        print("Correct Answer:", q.correct_answer)

except Exception as e:

    print("\nFAILED")
    print(type(e).__name__)
    print(str(e))


# -----------------------------
# Print Full JSON Output
# -----------------------------

print("\n=== FULL JSON OUTPUT ===")

print(parsed_mcqs.model_dump_json(indent=2))

MCQ file loaded successfully

=== TEST DETAILS ===
Title: General MCQs
Total Questions: 10

=== SAMPLE QUESTIONS ===

Question 1:
Which data structure follows the LIFO principle?
A. Queue
B. Stack
C. Array
D. Linked List
Correct Answer: B

Question 2:
Which keyword is used to create a class in Java?
A. object
B. struct
C. class
D. define
Correct Answer: C

Question 3:
What is the time complexity of binary search in a sorted array?
A. O(n)
B. O(log n)
C. O(n log n)
D. O(1)
Correct Answer: B

=== FULL JSON OUTPUT ===
{
  "title": "General MCQs",
  "total_questions": 10,
  "mcqs": [
    {
      "question": "Which data structure follows the LIFO principle?",
      "options": [
        "A. Queue",
        "B. Stack",
        "C. Array",
        "D. Linked List"
      ],
      "correct_answer": "B"
    },
    {
      "question": "Which keyword is used to create a class in Java?",
      "options": [
        "A. object",
        "B. struct",
        "C. class",
        "D. define"
      ],
   